# URBAN PULSE – Data Preparation

This notebook prepares the NYC 311 service-request data for analysis and MySQL storage.

The steps are:
1. Load the raw CSV file.
2. Keep the columns needed for this project.
3. clean dates, text values and ZIP codes.
4. Check data quality without deleting meaningful missing values.
5. Create analysis columns.
6. Save the cleaned datasets.

Run the cells from top to bottom. This notebook expects to be stored in the `notebooks` folder.

In [ ]:
import os
import numpy as np
import pandas as pd

## 1. Load the source data

The raw file is kept unchanged. All cleaning is performed on a separate DataFrame.

In [ ]:
raw_df = pd.read_csv("../data/raw/complaints.csv")

print("Original dataset shape:", raw_df.shape)

## 2. Select the project columns

Only the 13 fields required for the project are retained. This keeps the working dataset focused.

In [ ]:
required_columns = [
    "unique_key",
    "created_date",
    "closed_date",
    "due_date",
    "agency",
    "complaint_type",
    "borough",
    "status",
    "latitude",
    "longitude",
    "incident_zip",
    "location_type",
    "descriptor"
]

cleaned_df = raw_df[required_columns].copy()

print("Selected dataset shape:", cleaned_df.shape)

## 3. Clean date and text columns

Dates are converted to pandas datetime values. Invalid date text becomes a missing value so it can be reviewed. Text is stripped of leading and trailing spaces.

In [ ]:
date_columns = ["created_date", "closed_date", "due_date"]

for column in date_columns:
    cleaned_df[column] = pd.to_datetime(
        cleaned_df[column],
        errors="coerce"
    )

text_columns = [
    "agency",
    "complaint_type",
    "borough",
    "status",
    "location_type",
    "descriptor"
]

for column in text_columns:
    cleaned_df[column] = cleaned_df[column].str.strip()

print("Date columns converted.")
print("Text columns cleaned.")

## 4. Clean ZIP codes and check coordinates

ZIP codes are stored as text because they are identifiers, not values for arithmetic. Latitude and longitude are checked against their valid numeric ranges.

In [ ]:
cleaned_df["incident_zip"] = (
    cleaned_df["incident_zip"]
    .astype("Int64")
    .astype("string")
)

invalid_latitude = (
    (cleaned_df["latitude"] < -90) |
    (cleaned_df["latitude"] > 90)
)

invalid_longitude = (
    (cleaned_df["longitude"] < -180) |
    (cleaned_df["longitude"] > 180)
)

print("Invalid latitude values:", invalid_latitude.sum())
print("Invalid longitude values:", invalid_longitude.sum())

## 5. Handle missing descriptive values

Missing borough, ZIP, location type and descriptor values are labelled so they remain visible in grouped analysis. Missing closure dates and due dates are left missing because they have a real meaning in this dataset.

In [ ]:
cleaned_df["borough"] = cleaned_df["borough"].fillna("Unknown")
cleaned_df["incident_zip"] = cleaned_df["incident_zip"].fillna("Unknown")
cleaned_df["location_type"] = cleaned_df["location_type"].fillna("Unknown")
cleaned_df["descriptor"] = cleaned_df["descriptor"].fillna("Not Provided")

## 6. Create analysis fields

Resolution time is calculated only when both dates are present and the closure date is not earlier than the creation date. The original dates are retained, and invalid date conditions are recorded in separate flags.

In [ ]:
valid_closure = (
    cleaned_df["created_date"].notna() &
    cleaned_df["closed_date"].notna() &
    (cleaned_df["closed_date"] >= cleaned_df["created_date"])
)

cleaned_df["resolution_hours"] = np.nan

cleaned_df.loc[valid_closure, "resolution_hours"] = (
    cleaned_df.loc[valid_closure, "closed_date"] -
    cleaned_df.loc[valid_closure, "created_date"]
).dt.total_seconds() / 3600

cleaned_df["is_closed"] = cleaned_df["closed_date"].notna()

cleaned_df["created_year"] = cleaned_df["created_date"].dt.year
cleaned_df["created_month"] = cleaned_df["created_date"].dt.month
cleaned_df["created_weekday"] = cleaned_df["created_date"].dt.day_name()
cleaned_df["created_hour"] = cleaned_df["created_date"].dt.hour

cleaned_df["invalid_closure_date_flag"] = (
    cleaned_df["closed_date"].notna() &
    (cleaned_df["closed_date"] < cleaned_df["created_date"])
)

cleaned_df["invalid_due_date_flag"] = (
    cleaned_df["due_date"].notna() &
    (cleaned_df["due_date"] < cleaned_df["created_date"])
)

cleaned_df["invalid_latitude_flag"] = (
    cleaned_df["latitude"].notna() &
    ((cleaned_df["latitude"] < -90) | (cleaned_df["latitude"] > 90))
)

cleaned_df["invalid_longitude_flag"] = (
    cleaned_df["longitude"].notna() &
    ((cleaned_df["longitude"] < -180) | (cleaned_df["longitude"] > 180))
)

## 7. Review the cleaned dataset

These checks summarize the result before saving. Missing values in closure and due dates are not automatically errors.

In [ ]:
print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))
print("Duplicate request IDs:", cleaned_df["unique_key"].duplicated().sum())

print("\nMissing values:")
print(cleaned_df.isna().sum())

print("\nValid resolution-time records:", cleaned_df["resolution_hours"].notna().sum())
print("Invalid closure-date flags:", cleaned_df["invalid_closure_date_flag"].sum())
print("Invalid due-date flags:", cleaned_df["invalid_due_date_flag"].sum())

## 8. Save the cleaned datasets

`complaints_cleaned.csv` contains the selected and cleaned source fields. `complaints_mysql_ready.csv` also includes the derived fields used in SQL analysis.

In [ ]:
os.makedirs("../data/cleaned", exist_ok=True)

cleaned_df[required_columns].to_csv(
    "../data/cleaned/complaints_cleaned.csv",
    index=False
)

cleaned_df.to_csv(
    "../data/cleaned/complaints_mysql_ready.csv",
    index=False
)

print("Cleaned datasets saved.")

## Preparation summary

- The raw source file was kept unchanged.
- The project dataset was reduced to the 13 selected fields before feature creation.
- Date and text fields were cleaned.
- ZIP codes were kept as text.
- Missing closure and due dates were preserved.
- Resolution duration and date/time features were created.
- Data-quality flags were added instead of silently deleting questionable records.

The MySQL import process is handled separately from this data-preparation notebook.